# HuggingFace Transformers — Using Pre-trained Models

Training LLMs from scratch takes weeks on hundreds of GPUs.
Instead, you use **pre-trained models** from HuggingFace and adapt them.

**Key classes:**
- `AutoTokenizer` — converts text to token IDs
- `AutoModel` — the raw model (outputs hidden states)
- `AutoModelForCausalLM` — model with language modeling head (outputs logits → text generation)
- `AutoModelForSequenceClassification` — model with classification head

**Install:** `pip install transformers`

In [ ]:
# pip install transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# --- Tokenizer ---
# Converts text → token IDs (integers) → model input
# Each model has its own tokenizer (vocab, special tokens differ)

tokenizer = AutoTokenizer.from_pretrained('gpt2')

text   = 'The quick brown fox jumps over the lazy dog'
tokens = tokenizer(text, return_tensors='pt')   # pt = PyTorch tensors

print('Input IDs:  ', tokens['input_ids'])
print('Shape:      ', tokens['input_ids'].shape)
print('Tokens:     ', tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))
print('Decoded:    ', tokenizer.decode(tokens['input_ids'][0]))

In [ ]:
# --- Load model and run inference ---

model = AutoModelForCausalLM.from_pretrained('gpt2')
model.eval()

print(f'GPT-2 parameters: {sum(p.numel() for p in model.parameters()):,}')

# Forward pass — get logits
with torch.no_grad():
    outputs = model(**tokens)

print('Logits shape:', outputs.logits.shape)  # [1, seq_len, vocab_size]
# The model predicts the NEXT token at every position

In [ ]:
# --- Text generation ---

input_ids = tokenizer.encode('Once upon a time', return_tensors='pt')

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=True,       # sampling (creative) vs greedy (deterministic)
        temperature=0.8,      # lower = more focused, higher = more random
        top_p=0.9             # nucleus sampling — only sample from top 90% probability mass
    )

generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print('Generated text:')
print(generated)

In [ ]:
# --- Using pipeline (highest-level API) ---
from transformers import pipeline

# Sentiment analysis
classifier = pipeline('sentiment-analysis')
result = classifier('I absolutely love this product!')
print('Sentiment:', result)

# Text generation
generator = pipeline('text-generation', model='gpt2')
result = generator('In the future, AI will', max_new_tokens=30, num_return_sequences=1)
print('Generated:', result[0]['generated_text'])